In [30]:
import pandas as pd

df = pd.read_csv("Installment_shorter_sampled.csv")

# print(df.shape)
# print(df.head())
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 11894 entries, 0 to 11893
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Asset ID          11894 non-null  int64
 1   Origination Date  11894 non-null  str  
 2    Total Advance    11894 non-null  str  
 3    Total EMI        11894 non-null  str  
 4   Payment Date      11894 non-null  str  
 5    Payment Amount   11894 non-null  str  
dtypes: int64(1), str(5)
memory usage: 557.7 KB
None


### Check Duplicates


In [31]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)
df.head()

Duplicate rows: 638


,Asset ID,Origination Date,Total Advance,Total EMI,Payment Date,Payment Amount
0,264,1/31/2024,"17,160","20,540",2/7/2024,860
1,264,1/31/2024,"17,160","20,540",2/14/2024,860
2,264,1/31/2024,"17,160","20,540",2/21/2024,860
3,264,1/31/2024,"17,160","20,540",2/28/2024,860
4,264,1/31/2024,"17,160","20,540",3/6/2024,860


### Drop Duplicates


In [32]:
df = df.drop_duplicates()

print(df.shape)

(11256, 6)


### Convert & Check Datetime


In [ ]:
df["Origination Date"] = pd.to_datetime(
    df["Origination Date"],
    errors="coerce" # if any parsing error raise then it placed an NaT(Not a Date) instead of crashed
)
df["Payment Date"] = pd.to_datetime(
    df["Payment Date"],
    errors="coerce"
)

print(
    "Invalid Origination Dates:",
    df["Origination Date"].isna().sum()
)

print(
    "Invalid Payment Dates:",
    df["Payment Date"].isna().sum()
)

Invalid Origination Dates: 0
Invalid Payment Dates: 0


### Check abnormal Dates

- Payment date cannot be <= Origination Date


In [34]:
invalid_payments = (
    df["Payment Date"]
    < df["Origination Date"]
)

print(
    "Payments before origination:",
    invalid_payments.sum()
)

Payments before origination: 3


### Remove abnormal dates


In [35]:
df = df[
    df["Payment Date"]
    >= df["Origination Date"]
]

In [36]:
df.shape
df.columns
# df.info()

Index(['Asset ID', 'Origination Date', ' Total Advance ', ' Total EMI ',
       'Payment Date', ' Payment Amount '],
      dtype='str')

### Columns name contains extra space, its need to trim


In [37]:
df.columns = df.columns.str.strip()
df.columns

Index(['Asset ID', 'Origination Date', 'Total Advance', 'Total EMI',
       'Payment Date', 'Payment Amount'],
      dtype='str')

### Money Columns are still str format so we need to convert it to int


In [ ]:
money_cols = ["Total Advance", "Total EMI", "Payment Amount"]
for col in money_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce" #if any parsing error raise then it placed an NaN(Not a Number) instead of crashed
    )


### Check for missing numeric values


In [39]:
for col in money_cols:
    print(
        col,
        df[col].isna().sum()
    )

Total Advance 0
Total EMI 0
Payment Amount 4396


In [40]:
df.head()

,Asset ID,Origination Date,Total Advance,Total EMI,Payment Date,Payment Amount
0,264,2024-01-31,17160,20540,2024-02-07,860.0
1,264,2024-01-31,17160,20540,2024-02-14,860.0
2,264,2024-01-31,17160,20540,2024-02-21,860.0
3,264,2024-01-31,17160,20540,2024-02-28,860.0
4,264,2024-01-31,17160,20540,2024-03-06,860.0


### Remove Invalid Rows


In [44]:
df = df.dropna(
    subset=money_cols
)

In [45]:
df.info()

<class 'pandas.DataFrame'>
Index: 6857 entries, 0 to 11893
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Asset ID          6857 non-null   int64         
 1   Origination Date  6857 non-null   datetime64[us]
 2   Total Advance     6857 non-null   int64         
 3   Total EMI         6857 non-null   int64         
 4   Payment Date      6857 non-null   datetime64[us]
 5   Payment Amount    6857 non-null   float64       
dtypes: datetime64[us](2), float64(1), int64(3)
memory usage: 375.0 KB


### Checking negative or 0 values


In [48]:
for col in money_cols:
    print(
        col,
        (df[col] <= 0).sum()
    )

Total Advance 0
Total EMI 0
Payment Amount 490


In [49]:
df.to_csv(
    "cleaned_installments.csv",
    index=False
)